# Benchmark: ksenya MILP variants × task-load sweep

Сравнение 4-х вариантов ksenya-MILP на одном или нескольких перцентилях task-sweep датасета.

Алгоритмы:
- `milp_ksenya` — самый точный
- `milp_ksenya_decomp` — декомпозиция по ближайшему депо
- `milp_ksenya_knn` — k-NN arc-filter без декомпозиции
- `milp_ksenya_decomp_knn` — combo: декомпозиция + k-NN

На выходе — сводная таблица «перцентиль × алгоритм» с runtime, assigned/unassigned tasks, флагом «все задачи отданы» и transport-work.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
print("REPO_ROOT:", REPO_ROOT)

## Генерация датасетов


In [ ]:
import subprocess

BUILDER = REPO_ROOT / "demo/data/object_mass_feasible_task_sweep_5pct_fullfleet/scripts/build_task_sweep_fixed_fleet.py"
SOURCE  = REPO_ROOT / "demo/data/object_mass_feasible_fullfleet/dataset_real_spb_day_object_mass_feasible.json"
OUT_DIR = REPO_ROOT / "demo/data/object_mass_feasible_task_sweep_5pct_fullfleet"

need_build = not (OUT_DIR / "dataset_real_spb_day_object_mass_feasible_tasks_p005.json").exists()
if need_build:
    subprocess.run(
        ["python", str(BUILDER),
         "--source", str(SOURCE),
         "--out-dir", str(OUT_DIR)],
        check=True,
    )
    print("datasets built")
else:
    print("datasets already present, skipping build")

## Конфигурация прогона

`PERCENTILES` — какие перцентили берем. По дефолту 5/10/15% 

`ALGORITHMS` — 3 варианта с их kwargs.

Ориентировочные размеры RAM :
- `milp_ksenya` — реалистично только p005 (250 ГБ пик)
- `milp_ksenya_decomp` — p005 на грани (пик 22 ГБ)
- `milp_ksenya_knn` (k=10) — до p015 комфортно (6.6 ГБ на p005)
- `milp_ksenya_decomp_knn` (k=5) — до p030 sequential (648 МБ на p005, 2 ГБ на p030)

In [ ]:
DATA_DIR = REPO_ROOT / "demo" / "data" / "object_mass_feasible_task_sweep_5pct_fullfleet"
DATASET_NAME_PREFIX = "dataset_real_spb_day_object_mass_feasible_tasks"
SUMMARY_NAME_PREFIX = "summary_real_spb_day_object_mass_feasible_tasks"

#  "p020", "p025", …, "p100".
PERCENTILES = ["p005", "p010", "p015"]

ALGORITHMS = [
    ("milp_ksenya", {}),
    ("milp_ksenya_decomp", {
        "time_limit_sec": 600,
        "mip_rel_gap": 0.05,
        "verbose": True,
    }),
    ("milp_ksenya_knn", {
        "knn_k": 10,
        "time_limit_sec": 600,
        "mip_rel_gap": 0.05,
        "verbose": True,
    }),
    ("milp_ksenya_decomp_knn", {
        "knn_k": 5,
        "time_limit_sec": 600,
        "mip_rel_gap": 0.05,
        "verbose": True,
    }),
]

SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = False  # True — видны внутренние emit'ы runner; False — тихо.

missing = [p for p in PERCENTILES if not (DATA_DIR / f"{DATASET_NAME_PREFIX}_{p}.json").exists()]
if missing:
    raise FileNotFoundError(
        f"Нет файлов для перцентилей: {missing}.\n"
        f"Сгенерируй их скриптом build_task_sweep_fixed_fleet.py."
    )
print("Перцентили:", PERCENTILES)
print("Алгоритмы:", [a for a, _ in ALGORITHMS])

## Прогон

Для каждой пары (перцентиль, алгоритм) вызывается `execute_solver` и собираются метрики. Ошибки (OOM/таймаут/исключения) перехватываются и пишутся в колонку `error`, чтобы прогон не падал целиком.

In [ ]:
from flowopt.pipeline_runtime import execute_solver


def run_one(pct: str, algorithm: str, params: dict) -> dict:
    dataset_path = DATA_DIR / f"{DATASET_NAME_PREFIX}_{pct}.json"
    t0 = time.perf_counter()
    if SHOW_ALGO_PROGRESS:
        print(f"[start] {pct} / {algorithm}")
    try:
        execution = execute_solver(
            algorithm=algorithm,
            dataset_path=dataset_path,
            solver_kwargs=params,
            show_progress=SHOW_SOLVER_DETAILS,
            verbose=False,
        )
        m = execution.metrics.as_dict()
        elapsed = time.perf_counter() - t0
        row = {
            "pct": pct,
            "algorithm": algorithm,
            "runtime_sec": round(float(m.get("runtime_sec") or elapsed), 2),
            "feasible": bool(m.get("feasible")),
            "all_checks_ok": bool(m.get("all_checks_ok")),
            "assigned_routes": int(m.get("assigned_routes") or 0),
            "unassigned_tasks": int(m.get("unassigned_tasks") or 0),
            "all_tasks_assigned": int(m.get("unassigned_tasks") or 0) == 0,
            "active_agents": int(m.get("active_agents") or 0),
            "transport_work_ton_km": m.get("transport_work_ton_km"),
            "total_km": m.get("total_km"),
            "total_hours": m.get("total_hours"),
            "error": (m.get("details") or {}).get("solver_error"),
        }
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        row = {
            "pct": pct,
            "algorithm": algorithm,
            "runtime_sec": round(elapsed, 2),
            "feasible": False,
            "all_checks_ok": False,
            "assigned_routes": 0,
            "unassigned_tasks": None,
            "all_tasks_assigned": False,
            "active_agents": 0,
            "transport_work_ton_km": None,
            "total_km": None,
            "total_hours": None,
            "error": f"{type(exc).__name__}: {exc}",
        }
    if SHOW_ALGO_PROGRESS:
        status = "OK" if row["all_tasks_assigned"] else ("ERR" if row["error"] else "PARTIAL")
        print(f"[done]  {pct} / {algorithm}: {row['runtime_sec']:.1f}s [{status}]")
    return row


rows = []
for pct in PERCENTILES:
    for algo, params in ALGORITHMS:
        rows.append(run_one(pct, algo, params))

raw_df = pd.DataFrame(rows)
raw_df

## Сводная таблица

Пивот «перцентиль × алгоритм». Для каждой клетки видно:
- runtime, сек
- все ли задачи отданы (Y/N)
- число неотданных задач
- transport-work (тонаж·км)

In [ ]:
def _yes_no(v) -> str:
    if v is True:
        return "✓"
    if v is False:
        return "✗"
    return "—"


raw_df_view = raw_df.copy()
raw_df_view["all_tasks_assigned"] = raw_df_view["all_tasks_assigned"].map(_yes_no)

summary_cols = ["runtime_sec", "all_tasks_assigned", "unassigned_tasks", "transport_work_ton_km"]
summary = raw_df_view.pivot(index="pct", columns="algorithm", values=summary_cols)
summary = summary.reorder_levels([1, 0], axis=1).sort_index(axis=1)

print("=== Summary: перцентиль × алгоритм ===\n")
summary

In [ ]:
# Отдельные узкие пивоты — удобнее читать, чем общую таблицу выше.
runtime_pivot = raw_df.pivot(index="pct", columns="algorithm", values="runtime_sec")
assigned_pivot = (
    raw_df.pivot(index="pct", columns="algorithm", values="all_tasks_assigned")
    .map(_yes_no)
)
unassigned_pivot = raw_df.pivot(index="pct", columns="algorithm", values="unassigned_tasks")
tw_pivot = raw_df.pivot(index="pct", columns="algorithm", values="transport_work_ton_km")

print("\n--- runtime, сек ---")
print(runtime_pivot.round(2).to_string())

print("\n--- все задачи отданы? ---")
print(assigned_pivot.to_string())

print("\n--- # неотданных задач ---")
print(unassigned_pivot.to_string())

print("\n--- transport_work_ton_km ---")
print(tw_pivot.round(2).to_string())

In [ ]:
# Сохранить CSV рядом с ноутбуком (удобно для отчёта).
OUT_CSV = REPO_ROOT / "demo" / "ksenya_sweep_benchmark_results.csv"
raw_df.to_csv(OUT_CSV, index=False)
print("saved:", OUT_CSV)

In [ ]:
# Опционально: ошибки по запускам, если были.
errors = raw_df[raw_df["error"].notna()][["pct", "algorithm", "runtime_sec", "error"]]
if errors.empty:
    print("Ошибок нет — все прогоны завершились.")
else:
    print("Ошибки:")
    print(errors.to_string(index=False))